In [14]:
import pandas as pd
import numpy as np
import category_encoders as ce
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from datetime import datetime
import time as time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

train_data = pd.read_csv("train.csv")
train_data['Date_Occurred'] = pd.to_datetime(train_data['Date_Occurred'])
train_data['Date_Reported'] = pd.to_datetime(train_data['Date_Reported'])
train_data['Reported-Occured'] = train_data['Date_Reported'] - train_data['Date_Occurred']
train_data['Reported-Occured'] = train_data['Reported-Occured'].dt.days
train_data.drop(columns=['Cross_Street','Modus_Operandi','Premise_Description','Area_Name','Weapon_Used_Code','Status_Description'],axis=1,inplace=True)
bins = ['00:00 - 05:59', '06:00-11:59', '12:00-17:59', '18:00 - 23:59']
labels = ['Night', 'Morning', 'Afternoon', 'Evening']
def convert_to_time(x):
  hours = int(x // 100)
  minutes = int(x % 100)
  return f"{hours:02d}:{minutes:02d}:00"

train_data['Time_Occurred'] = train_data['Time_Occurred'].apply(convert_to_time)
train_data['Time_Occurred']= pd.to_datetime(train_data['Time_Occurred'])


train_data['Time_Occurred']= pd.to_datetime(train_data['Time_Occurred'], format='%H:%M')
bins = [0, 6, 12, 18, 24]
labels = ['Night', 'Morning', 'Afternoon', 'Evening']


hour = train_data['Time_Occurred'].dt.hour


train_data['Time_Occurred_Label'] = pd.cut(hour, bins=bins, labels=labels, right=False, include_lowest=True)

train_data['Time_Occurred'] = pd.to_datetime(train_data['Time_Occurred']).dt.strftime('%H:%M')

train_data.drop('Time_Occurred',inplace=True,axis=1)

bins = [0,15,180,365,float('inf')]
labels = ['Within 15 days','15 days to 6 months','6 months to 1 year', 'greater than 1 year']
train_data['Reported_bins'] = pd.cut(train_data['Reported-Occured'], bins=bins, labels=labels, right=False, include_lowest=True)

train_data.drop(columns=['Latitude','Longitude','Date_Reported','Date_Occurred'],inplace=True,axis=1)
train_data.loc[train_data['Victim_Age'] < 0, 'Victim_Age'] = 0
train_data['Victim_Sex'] = train_data.groupby('Crime_Category')['Victim_Sex'].transform(lambda x: x.fillna(x.mode()[0]))

train_data['Weapon_Description'] = train_data.groupby('Crime_Category')['Weapon_Description'].transform(lambda x: x.fillna('unknown'))

train_data['Victim_Descent'] = train_data.groupby('Victim_Sex')['Victim_Descent'].transform(lambda x: x.fillna(x.mode()[0]))

train_data.info()

X = train_data.drop(columns=['Crime_Category'])  
y = train_data['Crime_Category']  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X = train_data.drop(columns=['Crime_Category'])  # Replace with actual target column name
y = train_data['Crime_Category']

# Encode 'Crime_Category' target variable with Label Encoding
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y)  # Encode 'Crime_Category' (y_train)

# Ensure y_train_encoded is a Pandas Series
y_train_encoded = pd.Series(y_train_encoded)

# List of categorical columns that need encoding
categorical_cols = ['Location', 'Victim_Sex', 'Weapon_Description', 'Premise_Code', 'Victim_Descent', 'Status', 'Reporting_District_no', 'Time_Occurred_Label', 'Reported_bins']

# Apply target encoding for categorical columns in X_train (except the target variable itself)
target_encoder = ce.TargetEncoder(cols=categorical_cols)

# Ensure that X_train is a DataFrame
X_train = pd.DataFrame(X)


# Apply target encoding
X_train_encoded = target_encoder.fit_transform(X_train, y_train_encoded)

# Split data into train and test sets
X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(X_train_encoded, y_train_encoded, test_size=0.2, random_state=42)

# Create the Decision Tree classifier
clf = DecisionTreeClassifier(random_state=42)

# Fit the model on the training data
clf.fit(X_train_split, y_train_split)

# Make predictions on the test data
y_pred = clf.predict(X_test_split)

# Evaluate the model
accuracy = accuracy_score(y_test_split, y_pred)
print(f'Accuracy: {accuracy:.4f}')


C:\Users\Laddu\AppData\Local\Temp\ipykernel_7184\3948084673.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train_data['Date_Occurred'] = pd.to_datetime(train_data['Date_Occurred'])
C:\Users\Laddu\AppData\Local\Temp\ipykernel_7184\3948084673.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train_data['Date_Reported'] = pd.to_datetime(train_data['Date_Reported'])
C:\Users\Laddu\AppData\Local\Temp\ipykernel_7184\3948084673.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train_data['Time_Occurred']= pd.to_datetime(train_data['Time_Occurred'])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   Location               20000 non-null  object  
 1   Area_ID                20000 non-null  float64 
 2   Reporting_District_no  20000 non-null  float64 
 3   Part 1-2               20000 non-null  float64 
 4   Victim_Age             20000 non-null  float64 
 5   Victim_Sex             20000 non-null  object  
 6   Victim_Descent         20000 non-null  object  
 7   Premise_Code           20000 non-null  float64 
 8   Weapon_Description     20000 non-null  object  
 9   Status                 20000 non-null  object  
 10  Crime_Category         20000 non-null  object  
 11  Reported-Occured       20000 non-null  int64   
 12  Time_Occurred_Label    20000 non-null  category
 13  Reported_bins          20000 non-null  category
dtypes: category(2), float64(5), int64(1), 

In [15]:
import joblib

joblib.dump(clf, 'crime_classifier.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
joblib.dump(target_encoder, 'target_encoder.pkl')

['target_encoder.pkl']

In [10]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   Location               20000 non-null  object  
 1   Area_ID                20000 non-null  float64 
 2   Reporting_District_no  20000 non-null  float64 
 3   Part 1-2               20000 non-null  float64 
 4   Victim_Age             20000 non-null  float64 
 5   Victim_Sex             20000 non-null  object  
 6   Victim_Descent         20000 non-null  object  
 7   Premise_Code           20000 non-null  float64 
 8   Weapon_Description     20000 non-null  object  
 9   Status                 20000 non-null  object  
 10  Reported-Occured       20000 non-null  int64   
 11  Time_Occurred_Label    20000 non-null  category
 12  Reported_bins          20000 non-null  category
dtypes: category(2), float64(5), int64(1), object(5)
memory usage: 1.7+ MB


In [13]:
train_data = pd.read_csv("train.csv")
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Location               20000 non-null  object 
 1   Cross_Street           3448 non-null   object 
 2   Latitude               20000 non-null  float64
 3   Longitude              20000 non-null  float64
 4   Date_Reported          20000 non-null  object 
 5   Date_Occurred          20000 non-null  object 
 6   Time_Occurred          20000 non-null  float64
 7   Area_ID                20000 non-null  float64
 8   Area_Name              20000 non-null  object 
 9   Reporting_District_no  20000 non-null  float64
 10  Part 1-2               20000 non-null  float64
 11  Modus_Operandi         17259 non-null  object 
 12  Victim_Age             20000 non-null  float64
 13  Victim_Sex             17376 non-null  object 
 14  Victim_Descent         17376 non-null  object 
 15  Pr

In [13]:
X['Victim_Age'].min()

np.float64(0.0)